<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-04/week4_tool_use.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4 (starter): Multi-Tool Assistant

Everything below runs with no API key. The three tools, the validator, the dispatcher, and the dispatch loop are a **worked example** in a toy domain (arithmetic and unit conversion), driven by a scripted list of tool calls. They are the reference, not your submission.

Build your assistant in a domain you choose. The **TODO (you)** comments cover Parts 1 to 4; Part 5 is the submission checklist. The example uses only the Python standard library. For live calls through the OpenAI-compatible endpoint, install the client with `pip install openai`. Setting a key alone does not enable live calls.

In [ ]:
import os, ast, operator, json, math
HAS_API_KEY = bool(os.environ.get('GEMINI_API_KEY', '').strip())
print('GEMINI_API_KEY set:', HAS_API_KEY)
# The scripted loop below runs either way. Wiring the live model call is yours (Part 1).

In [ ]:
# Local function definitions with JSON schemas. Adapt these to your model API's tool format.
TOOLS = [
  {'name':'calculator','description':'Evaluate a basic arithmetic expression.',
   'parameters':{'type':'object','properties':{'expression':{'type':'string'}},'required':['expression'],'additionalProperties':False}},
  {'name':'unit_convert','description':'Convert a length between units.',
   'parameters':{'type':'object','properties':{'value':{'type':'number'},
       'from_unit':{'type':'string','enum':['m','km','mi','ft']},'to_unit':{'type':'string','enum':['m','km','mi','ft']}},
     'required':['value','from_unit','to_unit'],'additionalProperties':False}},
  {'name':'run_python','description':'Run a single allowlisted arithmetic expression (guarded code-runner).',
   'parameters':{'type':'object','properties':{'expression':{'type':'string'}},'required':['expression'],'additionalProperties':False}},
]
print('tools:', [t['name'] for t in TOOLS])
# TODO (you): replace these three with tools for a domain you pick. Keep the constraint
# discipline: enums where the value set is closed, required fields, explicit types.

In [ ]:
# Arithmetic-only interpreter. The full pre-execution check and time limit are TODOs (Part 2).
_OPS = {ast.Add:operator.add, ast.Sub:operator.sub, ast.Mult:operator.mul, ast.Div:operator.truediv, ast.Pow:operator.pow, ast.USub:operator.neg}
def _safe(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int,float): return node.value
    if isinstance(node, ast.BinOp): return _OPS[type(node.op)](_safe(node.left), _safe(node.right))
    if isinstance(node, ast.UnaryOp): return _OPS[type(node.op)](_safe(node.operand))
    raise ValueError('only arithmetic is allowed')   # blocks names, calls, imports, attributes
def calculator(expression): return _safe(ast.parse(expression, mode='eval').body)
_M = {'m':1.0,'km':1000.0,'mi':1609.344,'ft':0.3048}
def unit_convert(value, from_unit, to_unit): return value*_M[from_unit]/_M[to_unit]
def run_python(expression): return calculator(expression)   # guarded: same arithmetic-only evaluator
IMPL = {'calculator':calculator,'unit_convert':unit_convert,'run_python':run_python}

class ToolArgError(Exception): pass
# Validates the flat string/number schemas above. Extend this for your own schema types.
def validate(name, args):
    spec = next((t['parameters'] for t in TOOLS if t['name']==name), None)
    if spec is None: raise ToolArgError(f'unknown tool: {name}')
    if not isinstance(args, dict): raise ToolArgError('arguments must be a JSON object')
    for r in spec.get('required',[]):
        if r not in args: raise ToolArgError(f'missing required field: {r}')
    for k,v in args.items():
        p = spec['properties'].get(k)
        if p is None: raise ToolArgError(f'unexpected field: {k}')
        if p['type'] == 'string':
            if not isinstance(v, str): raise ToolArgError(f'{k} must be a string')
        elif p['type'] == 'number':
            if type(v) not in (int, float): raise ToolArgError(f'{k} must be a number (not a boolean)')
            if isinstance(v, float) and not math.isfinite(v): raise ToolArgError(f'{k} must be finite')
        else:
            raise ValueError(f'extend validate() to support schema type: {p["type"]}')
        if 'enum' in p and v not in p['enum']: raise ToolArgError(f'{k}={v!r} not in {p["enum"]}')
def dispatch(name, args):
    try:
        validate(name, args)
        output = IMPL[name](**args)
        json.dumps(output, allow_nan=False)  # Reject results the model cannot receive as JSON.
        return {'ok':True,'tool':name,'output':output}
    except ToolArgError as e:
        return {'ok':False,'tool':name,'error_type':'invalid_arguments','message':str(e)}
    except Exception as e:
        return {'ok':False,'tool':name,'error_type':'execution_error','message':str(e)}
print('happy path:', dispatch('calculator', {'expression':'3*(4+1)'}))
print('guarded:', dispatch('run_python', {'expression':'__import__("os").system("echo hi")'}))
# TODO (you), Part 2: run_python shares the arithmetic interpreter above; it has no time limit.
# For your code-runner, check the whole input against an allowlist before evaluation and
# enforce a time limit before connecting model-generated inputs. Arithmetic can exhaust resources.
# State what your guard permits and what it blocks: filesystem, network, process execution.

## Parts 1, 3, and 4: dispatch, evaluation, and recovery
The list below sends an invalid unit, then retries the same conversion with a valid unit. Both calls are scripted. No model chooses these calls or reads the errors, even when a key is set. Use this dispatch example to build your model loop, then evaluate your own tools and document a failure with recovery.

In [ ]:
scripted = [
  ('unit_convert', {'value':10,'from_unit':'klicks','to_unit':'mi'}),  # scripted invalid enum
  ('unit_convert', {'value':10,'from_unit':'km','to_unit':'mi'}),  # scripted corrected retry
  ('calculator',   {'expression':'2**10'}),
  ('run_python',   {'expression':'6*7'}),
]
for name, args in scripted:
    r = dispatch(name, args)
    tag = 'OK ' if r['ok'] else 'ERR'
    print(f'[{tag}] {name}({args}) -> {json.dumps(r, allow_nan=False)}')
# TODO (you), Part 1: send the tools and query to the model, execute its tool calls, return
# the results with matching call IDs, and let the model continue until it gives an answer.
# TODO (you), Part 3: run at least three queries covering every tool. Include a query that
# needs two tools in sequence, where the second uses the first result. Show the call logs.
# TODO (you), Part 4: find a real failure in your own model's calls. Show the schema, bad
# call, cause, and successful recovery. Explain whether a schema change, description, or retry fixed it.

## Part 5: Submit
Open a pull request with your schema design write-up, a link to your notebook, and a link to an issue documenting the failure and recovery. Describe your code-runner's allowlist, time limit, and blocked operations. Rubric: schemas (20), loop including a two-step sequence (25), guarded code-runner (20), failure with recovery (20), PR hygiene (15).